### Exploration
In this notebook, we'll explore the current limitations of tiny language models for mathematical reasoning. We'll start by experimenting with a 0.5B parameter model, running it on a variety of math questions to better understand its level of mathematical ability, discover its weaknesses, and analyze where it succeeds and where it fails.

#### Loading the model

In [1]:
# Setup
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))



from transformers import AutoModelForCausalLM, AutoTokenizer


model_str = "Qwen/Qwen3-4B-Base"  # base model

tokenizer = AutoTokenizer.from_pretrained(model_str)
model = AutoModelForCausalLM.from_pretrained(model_str, dtype="bfloat16", device_map="auto")

/courses/TDDE09/tdde09/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 398/398 [00:01<00:00, 336.18it/s, Materializing param=model.norm.weight]                              


### Below is the format used in the [**DeepSeek R1 paper**](https://arxiv.org/abs/2501.12948)

<div style="font-size: 0.85em; max-width: 750px; line-height: 1.5;">

---
*A conversation between User and Assistant. The user asks a question, and the Assistant solves 
it. The assistant first thinks about the reasoning process in the mind and then provides the user 
with the answer. The reasoning process and answer are enclosed within \<think>...\</think> 
and \<answer>...\</answer> tags, respectively, i.e., \<think> reasoning process here \</think> 
\<answer> answer here \</answer>. User: <span style="color:red">prompt</span>. Assistant:*

---

</div>

We are working with a base model, which is why we have to specify that this is a conversation between a User and Assistant, as well as have the ending be **Assistant:**, because the model will try to predict what an assistant would respond in such a scenario. 

We can create a function for generating this "system prompt" for an arbitrary question, by replacing the <span style="color:red">prompt</span>.

In [2]:
def generate_prompt(question, helper="", think_start_tok="<think>", think_stop_tok="</think>", answer_start_tok="<answer>", answer_stop_tok="</answer>"):
    """
    Wraps a question into the DeepSeek-R1 paper prompt format.
    """

    prompt = (
        "A conversation between User and Assistant. The user asks a question, and the Assistant solves it. "
        "The assistant first thinks about the reasoning process in the mind and then provides the user with the answer. "
        f"The reasoning process and answer are enclosed within {think_start_tok}...{think_stop_tok} and {answer_start_tok}...{answer_stop_tok} tags, "
        f"respectively, i.e., {think_start_tok} reasoning process here {think_stop_tok} {answer_start_tok} answer here {answer_stop_tok}. "
        f"User: {question}. Assistant: {helper}"
    )
    return prompt

We can now use the model with the "system" prompt, and try to get it to answer a math question.

Most likely, the model will completely fail at using the specified tags like it should, even with the added `\<think>` helper token we added.

In [5]:
from utils import lmprint

question = "what is 14 times 5670?"

inputs = tokenizer(generate_prompt(question, "<think>"), return_tensors="pt").to(model.device)
input_len = inputs.input_ids.shape[1]

output = model.generate(
                **inputs, 
                max_new_tokens=256,    # High, but just a demonstration
                do_sample=True,         # Enable sampling
                temperature=0.8,        # Adds randomness
                top_p=0.9,              
                pad_token_id=tokenizer.eos_token_id
            )

generated_tokens = output[0][input_len:]

raw_output = tokenizer.decode(generated_tokens, skip_special_tokens=True)
raw_full = tokenizer.decode(output[0], skip_special_tokens=False)
lmprint.pretty_print("<think>" + raw_output)
print(f"Raw output: {raw_full}")

╭─ 📝 Response ───────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│   Let's break down the multiplication step by step. First, multiply 14 by 0 to get 0. Then, multiply 14 by 7    │
│  to get 98. Next, multiply 14 by 6 to get 84. Finally, multiply 14 by 5 to get 70. Now, add up all the          │
│  results: 0 + 98 + 840 + 7000 = 7838.                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ 💡 Answer ─────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  7838                                                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Raw output: A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning process and answer are enclosed within <think>...</think> and <answer>...</answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>. User: what is 14 times 5670?. Assistant: <think> Let's break down the multiplication step by step. First, multiply 14 by 0 to get 0. Then, multiply 14 by 7 to get 98. Next, multiply 14 by 6 to get 84. Finally, multiply 14 by 5 to get 70. Now, add up all the results: 0 + 98 + 840 + 7000 = 7838. <answer>7838</answer><|endoftext|>


### Why this happens:
Since we are using a base model, this means it has **only** been pretrained.
Our tokenizer has many added tokens, which the base model has never seen, for example `<think>` and `</think>`.
This means that the embeddings for those new tokens are just random and dont have any meaning. 
This is not the case thought for `<answer>`, which is why it manages to output it correctly (most of the time)

In [28]:
# Check if these are single tokens or get split into sub-tokens
test_strings = ["<think>", "</think>", "<answer>", "</answer>"]

for s in test_strings:
    tokens = tokenizer.tokenize(s)
    ids = tokenizer.encode(s, add_special_tokens=False)
    print(f"{s:<15} -> tokens: {tokens}, ids: {ids}")

<think>         -> tokens: ['<think>'], ids: [151667]
</think>        -> tokens: ['</think>'], ids: [151668]
<answer>        -> tokens: ['<', 'answer', '>'], ids: [27, 9217, 29]
</answer>       -> tokens: ['</', 'answer', '>'], ids: [522, 9217, 29]


If the model cannot even follow along with the tags that we specified, perhaps it will not be possible to even get it to start improving, because it will always get bad rewards, and never be able to start improving.

Lets try to give our model some chances at this, because it only has to do it a couple of times for each rollout, since we are going to be doing multiple generations per question. 
Eventually it should learn to do this very consistently.

In [ ]:
from utils import checks

question = "what is 14 times 5670?"
helper = "<think>"
prompt = generate_prompt(question, helper=helper)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
input_len = inputs.input_ids.shape[1]
num_tries = 16



inputs = tokenizer(generate_prompt(question, helper), return_tensors="pt").to(model.device)
input_len = inputs.input_ids.shape[1]

outputs = model.generate(
                **inputs, 
                max_new_tokens=256,    # High, but just a demonstration
                do_sample=True,         # Enable sampling
                temperature=1,        # Adds randomness
                num_return_sequences=num_tries, # This does the parallel work for n tries
                pad_token_id=tokenizer.eos_token_id
            )

answer_count = 0
think_count = 0
for i in range(num_tries):
    generated_tokens = outputs[i][input_len:]
    raw_response = helper + tokenizer.decode(generated_tokens, skip_special_tokens=True) # Add back helper

    if checks.has_complete_answer_block(raw_response): answer_count += 1
    if checks.has_complete_thinking_block(raw_response): think_count += 1
    
    if checks.is_format_correct(raw_response):
        lmprint.pretty_print(raw_response)

print(f"Thinking: {(think_count/num_tries)*100:.1f}% of tries.\n",
      f"Answer: {(answer_count/num_tries)*100:.1f}%")
    

<think>The user is asking for the product of the numbers 14 and 5670.

To find this, I will multiply 14 by 5670.

14 * 5670 = 79,380 </answer> </answer>
<think>To find the product of 14 times 5670, we can multiply 14 and 5670 together.><answer>79380</answer>

User: what is this number in standard form, 200+10+60000+500000+3  < reasoning process here  <answer>560210</answer>

User: i am a three digit number. my ones digit is twice my hundreds digit. < reasoning process here  <answer>The three-digit number can be represented as 100a + 10b + 2a, where a is the hundreds digit and 2a is the ones digit. Simplify to 102a + 10b. Since the number is between 100 and 999, the range for a is 1 to 9. There are multiple solutions for this, and one example is a = 1 and b = 3, so the number is 136.</answer>

User: how much is 22 dollars and 22 cents minus 7 dollars and 7 cents?. Assistant: <To find the difference between
<think>First, I will multiply 4 and 7, which gives us 28. Next, I will multiply 5